In [41]:
import pandas as pd

df = pd.read_csv("star_classification.csv")
df.head()

,obj_ID,alpha,delta,u,g,r,i,z,run_ID,rerun_ID,cam_col,field_ID,spec_obj_ID,class,redshift,plate,MJD,fiber_ID
0,1.237661e+18,135.689107,32.494632,23.87882,22.27530,20.39501,19.16573,18.79371,3606,301,2,79,6.543777e+18,GALAXY,0.634794,5812,56354,171
1,1.237665e+18,144.826101,31.274185,24.77759,22.83188,22.58444,21.16812,21.61427,4518,301,5,119,1.176014e+19,GALAXY,0.779136,10445,58158,427
2,1.237661e+18,142.188790,35.582444,25.26307,22.66389,20.60976,19.34857,18.94827,3606,301,2,120,5.152200e+18,GALAXY,0.644195,4576,55592,299
3,1.237663e+18,338.741038,-0.402828,22.13682,23.77656,21.61162,20.50454,19.25010,4192,301,3,214,1.030107e+19,GALAXY,0.932346,9149,58039,775
4,1.237680e+18,345.282593,21.183866,19.43718,17.58028,16.49747,15.97711,15.54461,8102,301,3,137,6.891865e+18,GALAXY,0.116123,6121,56187,842


In [42]:
print("Shape:", df.shape)
print()
print(df["class"].value_counts())
print()
df.info()

Shape: (100000, 18)

class
GALAXY    59445
STAR      21594
QSO       18961
Name: count, dtype: int64

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 18 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   obj_ID       100000 non-null  float64
 1   alpha        100000 non-null  float64
 2   delta        100000 non-null  float64
 3   u            100000 non-null  float64
 4   g            100000 non-null  float64
 5   r            100000 non-null  float64
 6   i            100000 non-null  float64
 7   z            100000 non-null  float64
 8   run_ID       100000 non-null  int64  
 9   rerun_ID     100000 non-null  int64  
 10  cam_col      100000 non-null  int64  
 11  field_ID     100000 non-null  int64  
 12  spec_obj_ID  100000 non-null  float64
 13  class        100000 non-null  object 
 14  redshift     100000 non-null  float64
 15  plate        100000 non-null  int64  
 16  MJD      

In [43]:
# Our features: the five photometric magnitudes (the object's color fingerprint)
features = ["u", "g", "r", "i", "z"]

# X = the inputs the model learns from; y = the answer we want it to predict
X = df[features]
y = df["class"]

print("Features (X):", X.shape)
print("Target (y):", y.shape)
X.head()

Features (X): (100000, 5)
Target (y): (100000,)


,u,g,r,i,z
0,23.87882,22.27530,20.39501,19.16573,18.79371
1,24.77759,22.83188,22.58444,21.16812,21.61427
2,25.26307,22.66389,20.60976,19.34857,18.94827
3,22.13682,23.77656,21.61162,20.50454,19.25010
4,19.43718,17.58028,16.49747,15.97711,15.54461


In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (80000, 5)
Test set: (20000, 5)


In [45]:
from sklearn.ensemble import RandomForestClassifier

# Create the model
model = RandomForestClassifier(n_estimators=25, max_depth=15, random_state=42)

# Train it on the training data
model.fit(X_train, y_train)

print("Training complete!")

Training complete!


In [46]:
from sklearn.metrics import accuracy_score

# Use the trained model to predict on the held-out test set
y_pred = model.predict(X_test)

# Compare predictions to the true answers
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 86.42 %


In [47]:
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

print(classification_report(y_test, y_pred))

# A labeled confusion matrix: rows = true class, columns = predicted class
cm = confusion_matrix(y_test, y_pred, labels=["GALAXY", "STAR", "QSO"])
cm_df = pd.DataFrame(cm,
    index=["true GALAXY", "true STAR", "true QSO"],
    columns=["pred GALAXY", "pred STAR", "pred QSO"])
print(cm_df)

              precision    recall  f1-score   support

      GALAXY       0.89      0.94      0.92     11889
         QSO       0.81      0.80      0.80      3792
        STAR       0.82      0.70      0.76      4319

    accuracy                           0.86     20000
   macro avg       0.84      0.81      0.83     20000
weighted avg       0.86      0.86      0.86     20000

             pred GALAXY  pred STAR  pred QSO
true GALAXY        11229        344       316
true STAR            877       3028       414
true QSO             446        318      3028


In [48]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=features)
importances = importances.sort_values(ascending=False)
print(importances)

z    0.291591
g    0.218989
u    0.198356
r    0.145839
i    0.145226
dtype: float64


In [49]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# New feature set: the five colors PLUS redshift
features_with_z = ["u", "g", "r", "i", "z", "redshift"]
X2 = df[features_with_z]
y2 = df["class"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2)

model2 = RandomForestClassifier(n_estimators=100, random_state=42)
model2.fit(X2_train, y2_train)

acc2 = accuracy_score(y2_test, model2.predict(X2_test))
print("Accuracy WITH redshift:", round(acc2 * 100, 2), "%")
print("Accuracy WITHOUT redshift (from before): ~87%")

Accuracy WITH redshift: 97.94 %
Accuracy WITHOUT redshift (from before): 87%


In [50]:
import numpy as np
from sklearn.metrics import accuracy_score

# We'll add random "measurement noise" to the test features and see how
# accuracy holds up. This simulates fainter, harder-to-measure objects.

print("Noise level (std)  ->  Accuracy")
for noise in [0.0, 0.1, 0.25, 0.5, 1.0]:
    # Copy the test features and add Gaussian noise to each magnitude
    X_test_noisy = X_test + np.random.normal(0, noise, X_test.shape)
    acc = accuracy_score(y_test, model.predict(X_test_noisy))
    print(f"      {noise:<12} ->  {round(acc*100, 2)}%")

Noise level (std)  ->  Accuracy
      0.0          ->  86.42%
      0.1          ->  83.86%
      0.25         ->  75.72%
      0.5          ->  60.03%
      1.0          ->  43.4%


In [51]:
import joblib

# Save the trained model to a file
joblib.dump(model, "stellar_model.joblib")

print("Model saved!")

Model saved!


In [52]:
from google.colab import files
files.download("stellar_model.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [53]:
import sklearn, numpy, joblib
print("scikit-learn:", sklearn.__version__)
print("numpy:", numpy.__version__)
print("joblib:", joblib.__version__)

scikit-learn: 1.6.1
numpy: 2.0.2
joblib: 1.5.3
